# 02 · Inheritance

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 60 minutes live | [01 · Classes and objects](01_classes_and_objects.ipynb) | [03 · Special methods](03_special_methods.ipynb) |

Before anyone trains a fancy model, a good data scientist asks: "what would a simple rule achieve?" The retention team already has rules of thumb ("month-to-month customers are risky"), and any model must beat them to be worth deploying. In this notebook we write several such **baseline models**. They all need the same evaluation code, and they differ only in how they predict. That is exactly the situation **inheritance** was made for.

> **Learning goals.** By the end you can:
>
> - write a subclass that inherits behaviour from a base class and overrides one method
> - use `super()` to extend a parent's `__init__` instead of copying it
> - read a class hierarchy (`isinstance`, `__mro__`) in your own and in library code
> - judge whether inheritance is the right tool, using the "is-a" test

**Before you start.** Notebook 01 (classes, `__init__`, methods, properties, `@classmethod`).

**Exercises:** Core (everyone), Stretch (if you finish early), Challenge (for experienced programmers).

**In this notebook**

1. The problem: two rules, duplicated code
2. A base class and subclasses
3. Overriding: same name, more specific behaviour
4. `super()`: extend the parent instead of replacing it
5. Looking up the chain: the MRO
6. When not to inherit
7. Exercises

---

## Setup

The same data preparation as in notebook 01, plus the `ConfusionMatrix` class we built there (with properties and the `from_predictions` constructor). This time we keep the original text columns such as `Contract`, because our rules use them directly.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")

customers = pd.read_csv(DATA_PATH)
customers["TotalCharges"] = pd.to_numeric(customers["TotalCharges"], errors="coerce").fillna(0.0)
customers["Churn"] = (customers["Churn"] == "Yes").astype(int)

X = customers.drop(columns=["customerID", "Churn"])
y = customers["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


class ConfusionMatrix:
    """Hits and misses of a binary classifier (1 = churn)."""

    def __init__(self, tn: int, fp: int, fn: int, tp: int):
        self.tn, self.fp, self.fn, self.tp = tn, fp, fn, tp

    @classmethod
    def from_predictions(cls, y_true, y_pred) -> "ConfusionMatrix":
        t, p = np.asarray(y_true), np.asarray(y_pred)
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def n(self) -> int:
        return self.tn + self.fp + self.fn + self.tp

    @property
    def accuracy(self) -> float:
        return (self.tp + self.tn) / self.n

    @property
    def precision(self) -> float:
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if p + r else 0.0

---

## 1. The problem: two rules, duplicated code

Here are two baselines written as independent classes. Read them and look for what is identical.

In [ ]:
class MajorityClassModel:
    """Predicts the most frequent class for everybody."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "MajorityClassModel":
        self.majority_class = int(y.mode()[0])
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return np.full(len(X), self.majority_class)

    def report(self, X: pd.DataFrame, y: pd.Series) -> str:
        cm = ConfusionMatrix.from_predictions(y, self.predict(X))
        return f"{'majority class':<22} accuracy={cm.accuracy:.2f} precision={cm.precision:.2f} recall={cm.recall:.2f}"


class ContractRuleModel:
    """Flags every month-to-month customer: the retention team's current rule of thumb."""

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "ContractRuleModel":
        return self  # a fixed rule learns nothing

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (X["Contract"] == "Month-to-month").astype(int).to_numpy()

    def report(self, X: pd.DataFrame, y: pd.Series) -> str:
        cm = ConfusionMatrix.from_predictions(y, self.predict(X))
        return f"{'month-to-month rule':<22} accuracy={cm.accuracy:.2f} precision={cm.precision:.2f} recall={cm.recall:.2f}"


for m in [MajorityClassModel(), ContractRuleModel()]:
    print(m.fit(X_train, y_train).report(X_test, y_test))

Two observations, one about data science and one about code:

- The majority-class model reaches about 73 % accuracy and finds **zero** churners. Accuracy is a misleading score when classes are imbalanced. The simple contract rule finds most churners, at the price of many false alarms.
- `report` is copied word for word. If we decide to add F1 to the report, we have to change every baseline. With five baselines, one of them will be forgotten.

## 2. A base class and subclasses

We move the shared code into a **base class** (also called parent or superclass) and let each baseline **inherit** from it. A **subclass** (child) is written `class Child(Parent):` and automatically has every method of the parent.

In [ ]:
class BaselineModel:
    """Common behaviour of all rule-based baselines."""

    def __init__(self, name: str | None = None):
        self.name = name or type(self).__name__

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "BaselineModel":
        return self  # most rules learn nothing; subclasses that do learn override this

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        raise NotImplementedError(f"{type(self).__name__} must implement predict()")

    def evaluate(self, X: pd.DataFrame, y: pd.Series) -> ConfusionMatrix:
        return ConfusionMatrix.from_predictions(y, self.predict(X))

    def report(self, X: pd.DataFrame, y: pd.Series) -> str:
        cm = self.evaluate(X, y)
        return (
            f"{self.name:<22} flagged={cm.tp + cm.fp:5d}  precision={cm.precision:.2f}  "
            f"recall={cm.recall:.2f}  f1={cm.f1:.2f}"
        )


class MajorityClassModel(BaselineModel):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "MajorityClassModel":
        self.majority_class = int(y.mode()[0])
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return np.full(len(X), self.majority_class)


class ContractRuleModel(BaselineModel):
    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (X["Contract"] == "Month-to-month").astype(int).to_numpy()


for m in [MajorityClassModel(), ContractRuleModel()]:
    print(m.fit(X_train, y_train).report(X_test, y_test))

Neither subclass defines `report` or `evaluate`, yet both have them: they are **inherited**. F1 now appears in every report because we added it in one place. `ContractRuleModel` does not even define `fit`; it uses the parent's version, which simply returns `self`.

Notice how `BaselineModel.predict` raises `NotImplementedError`: the base class says "every baseline must be able to predict, but I cannot know how". In notebook 04 you will meet a stricter tool for this (abstract base classes).

### `isinstance` follows the hierarchy

A subclass object **is also** an object of its parent class. Code that accepts a `BaselineModel` therefore accepts every subclass.

In [ ]:
rule = ContractRuleModel()
print(isinstance(rule, ContractRuleModel))
print(isinstance(rule, BaselineModel))
print(isinstance(rule, MajorityClassModel))
print(issubclass(ContractRuleModel, BaselineModel))

---

## 3. Overriding: same name, more specific behaviour

When a subclass defines a method with the same name as the parent, the subclass version wins. That is **overriding**. Python looks for a method on the object's own class first and only then on its parents.

`MajorityClassModel` overrides `fit` (it needs to learn the majority class) and `predict`. `ContractRuleModel` overrides only `predict`.

### Check your understanding

```python
class AlwaysChurn(BaselineModel):
    def predict(self, X):
        return np.ones(len(X), dtype=int)

    def report(self, X, y):
        return "everyone churns!"

m = AlwaysChurn()
print(m.evaluate(X_test, y_test).recall)
print(m.report(X_test, y_test))
```

What do the two lines print?

<details><summary><b>Show answer</b></summary>

`1.0` and `everyone churns!`. `evaluate` is inherited from `BaselineModel`, but inside it `self.predict(X)` calls the **subclass's** `predict`, because `self` is an `AlwaysChurn`. Flagging everybody catches every churner (recall 1.0, at terrible precision). `report` is overridden, so the subclass version runs. This is the central trick of inheritance: shared code in the parent calls methods that each child fills in.
</details>

---

## 4. `super()`: extend the parent instead of replacing it

The contract rule is hard-coded to month-to-month. The team might want to flag one-year contracts too. So the subclass needs a **parameter**, which means its own `__init__`. But the parent's `__init__` sets `self.name`, and we do not want to copy that logic. `super()` gives access to the parent's version of a method, so the child can run it and then add its own work.

In [ ]:
class ContractRuleModel(BaselineModel):
    def __init__(self, risky_contracts: tuple[str, ...] = ("Month-to-month",)):
        super().__init__(name=f"contract in {list(risky_contracts)}")
        self.risky_contracts = risky_contracts

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return X["Contract"].isin(self.risky_contracts).astype(int).to_numpy()


for contracts in [("Month-to-month",), ("Month-to-month", "One year")]:
    print(ContractRuleModel(contracts).report(X_test, y_test))

Adding one-year contracts catches almost every churner, but costs about 300 extra calls, most of them to customers who would have stayed: one-year customers rarely churn (11 % against 43 % for month-to-month). Precision drops from 0.43 to 0.34.

### Check your understanding

What happens if we forget the `super().__init__(...)` line?

```python
class ForgetfulRule(BaselineModel):
    def __init__(self, risky_contracts=("Month-to-month",)):
        self.risky_contracts = risky_contracts

    def predict(self, X):
        return X["Contract"].isin(self.risky_contracts).astype(int).to_numpy()

ForgetfulRule().report(X_test, y_test)
```

<details><summary><b>Show answer</b></summary>

`AttributeError: 'ForgetfulRule' object has no attribute 'name'`. Defining `__init__` in the subclass overrides the parent's `__init__` completely, so `self.name = ...` never runs. The error only appears later, in `report`, which makes it confusing. Rule: if a subclass defines `__init__`, it almost always calls `super().__init__(...)` first.
</details>

---

## 5. Looking up the chain: the MRO

How does Python decide which method to run? It searches a list of classes called the **method resolution order** (MRO): the object's own class first, then its parent, then the parent's parent, up to `object`, the root of every class in Python.

In [ ]:
for cls in ContractRuleModel.__mro__:
    print(cls.__name__)

This is also how you find your way around library code. scikit-learn's estimators have long MROs made of small parent classes; you will read some of them in notebook 06.

In [ ]:
from sklearn.linear_model import LogisticRegression

[cls.__name__ for cls in LogisticRegression.__mro__]

---

## 6. When not to inherit

Inheritance means "**is a**": a `ContractRuleModel` *is a* `BaselineModel`, and it can be used everywhere a `BaselineModel` is expected. Ask that question before you inherit.

- A churn dataset *is not* a DataFrame. It *has* a DataFrame. Writing `class ChurnDataset(pd.DataFrame)` would expose hundreds of DataFrame methods that make no sense for it, and pandas makes subclassing hard on purpose.
- A model that combines two rules *is not* both rules. It *has* two rules.

"Has a" relationships are expressed by storing one object inside another (**composition**). That is often the better tool, and it is a main topic of notebook 08. Inheritance is for sharing behaviour between genuinely similar things, as with our baselines.

---

## 7. Exercises

### Exercise 7.1
> **Core** · for everyone

The team suspects that **new customers** churn most. Write `TenureRuleModel(BaselineModel)` that flags customers whose `tenure` (months with the company) is below `max_tenure`.

- `__init__(self, max_tenure=12)` stores the parameter and calls `super().__init__` with a readable name such as `"tenure < 12"`;
- override `predict` only; `fit`, `evaluate` and `report` are inherited.

Then compare it with the contract rule. Which one would you give to the retention team, and why?

In [ ]:
# Exercise 7.1: your code here


class TenureRuleModel(BaselineModel):
    def __init__(self, max_tenure: int = 12):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 7.1: run after solving it
rule = TenureRuleModel(max_tenure=12)
assert rule.max_tenure == 12
assert isinstance(rule, BaselineModel)
preds = rule.predict(pd.DataFrame({"tenure": [0, 11, 12, 40]}))
assert list(preds) == [1, 1, 0, 0]
assert "12" in rule.name

for model in [ContractRuleModel(), TenureRuleModel(12), TenureRuleModel(6)]:
    print(model.report(X_test, y_test))
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class TenureRuleModel(BaselineModel):
    def __init__(self, max_tenure: int = 12):
        super().__init__(name=f"tenure < {max_tenure}")
        self.max_tenure = max_tenure

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        return (X["tenure"] < self.max_tenure).astype(int).to_numpy()
```

The tenure rule calls far fewer customers and is more precise, but it misses more churners than the contract rule. Which is better depends on what a call costs compared with a lost customer. We will put numbers on that in notebook 03.
</details>

### Exercise 7.2
> **Stretch** · if you finish early

Why 12 months? Instead of guessing, **learn** the cutoff from the training data. Write `LearnedTenureRuleModel(TenureRuleModel)` that overrides only `fit`: try every cutoff from 1 to 72 months and keep the one with the highest F1 score on the training data. `predict` must stay untouched (it is inherited from `TenureRuleModel`). Update `self.name` so the report shows the learned value.

In [ ]:
# Exercise 7.2: your code here


class LearnedTenureRuleModel(TenureRuleModel):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "LearnedTenureRuleModel":
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 7.2: run after solving it
learned = LearnedTenureRuleModel().fit(X_train, y_train)
assert "predict" not in LearnedTenureRuleModel.__dict__, (
    "predict should be inherited, not redefined"
)
assert 1 <= learned.max_tenure <= 72
assert learned.evaluate(X_train, y_train).f1 >= TenureRuleModel(12).evaluate(X_train, y_train).f1
print(f"learned cutoff: {learned.max_tenure} months")
print(learned.report(X_test, y_test))
print("All checks passed")

<details><summary><b>Show hint</b></summary>

Inside `fit`, loop over the candidates, set `self.max_tenure = candidate`, and call `self.evaluate(X, y).f1`. The inherited `predict` uses whatever `self.max_tenure` is at that moment. Keep the best one at the end.
</details>

<details><summary><b>Show solution</b></summary>

```python
class LearnedTenureRuleModel(TenureRuleModel):
    def fit(self, X: pd.DataFrame, y: pd.Series) -> "LearnedTenureRuleModel":
        best_f1, best_cutoff = -1.0, self.max_tenure
        for cutoff in range(1, 73):
            self.max_tenure = cutoff
            f1 = self.evaluate(X, y).f1
            if f1 > best_f1:
                best_f1, best_cutoff = f1, cutoff
        self.max_tenure = best_cutoff
        self.name = f"tenure < {best_cutoff} (learned)"
        return self
```

Note that the cutoff is chosen on the **training** data and judged on the test data. Choosing it on the test data would make the test score optimistic.
</details>

### Exercise 7.3
> **Challenge** · for experienced programmers

The team wants a rule that flags a customer if **any** of several rules fires (for example "month-to-month **or** tenure below 6").

1. A colleague proposes `class CombinedRule(ContractRuleModel, TenureRuleModel)`, using multiple inheritance. Explain why this is the wrong tool. (Consider: how many `predict` methods can the class have, which `__init__` runs, and what about three rules, or two tenure rules with different cutoffs?)
2. Implement `AnyRuleModel(BaselineModel)` with **composition** instead: it receives a list of baseline models in `__init__`, `fit` fits each of them, and `predict` flags a customer if at least one rule flags them.
3. Is `AnyRuleModel` itself a baseline (is-a), and does it have baselines (has-a)? What does that let you do?

In [ ]:
# Exercise 7.3: your code here


class AnyRuleModel(BaselineModel):
    def __init__(self, rules: list[BaselineModel]):
        raise NotImplementedError("Your code here")

In [ ]:
# Check for exercise 7.3: run after solving it
combo = AnyRuleModel([ContractRuleModel(), TenureRuleModel(6)])
combo.fit(X_train, y_train)
toy = pd.DataFrame({"Contract": ["Two year", "Month-to-month", "Two year"], "tenure": [3, 40, 40]})
assert list(combo.predict(toy)) == [1, 1, 0]
nested = AnyRuleModel([combo, TenureRuleModel(2)])
assert list(nested.fit(X_train, y_train).predict(toy)) == [1, 1, 0]
print(combo.report(X_test, y_test))
print("All checks passed")

<details><summary><b>Show solution</b></summary>

```python
class AnyRuleModel(BaselineModel):
    def __init__(self, rules: list[BaselineModel]):
        super().__init__(name=" OR ".join(rule.name for rule in rules))
        self.rules = rules

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "AnyRuleModel":
        for rule in self.rules:
            rule.fit(X, y)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        votes = np.column_stack([rule.predict(X) for rule in self.rules])
        return votes.max(axis=1)
```

1. With multiple inheritance a class still has exactly one `predict`: Python picks the first one in the MRO (`ContractRuleModel.predict`) and the tenure rule is silently ignored. Only one parent's `__init__` runs unless every class cooperates through `super()`, which these do not. And inheritance is fixed when the class is written: "two tenure rules with different cutoffs" or "any three rules" would each need a new class.
2. With composition, the rules are ordinary objects chosen at run time: any number, any mix, any parameters.
3. Both. `AnyRuleModel` *is a* `BaselineModel` (so it inherits `evaluate` and `report` and can be used wherever a baseline is expected) and it *has* baselines. Because it is one of the things it contains, combinations can be nested, as in the check cell. This is the **Composite** pattern; scikit-learn's `Pipeline` and `VotingClassifier` follow the same idea.
</details>

---

## Summary

- **Inheritance** lets a subclass reuse everything its base class defines: `class Child(Parent):`.
- **Overriding** replaces a parent's method with a more specific one. Shared code in the parent can call methods that each child provides (`evaluate` calls `self.predict`).
- `super().__init__(...)` runs the parent's initialisation; forget it and attributes go missing.
- Python finds methods along the **MRO**; `isinstance` and `__mro__` show you the hierarchy, in your code and in libraries.
- Use inheritance only for genuine **is-a** relationships. For **has-a**, store objects inside objects (composition).
- Data science lesson along the way: always compare against simple baselines, and do not trust accuracy on imbalanced data.

**Common pitfalls:** defining `__init__` in a subclass without calling `super().__init__`; deep hierarchies where nobody remembers which parent provides what; inheriting from a class (like `pd.DataFrame`) just to reuse a few of its methods.

## Going further

- scikit-learn's own baselines: `sklearn.dummy.DummyClassifier` (we read its source in notebook 06).

---

## Sources and further reading

**Sources.** This notebook is based on:

- [The Python Tutorial, section 9.5: Inheritance](https://docs.python.org/3/tutorial/classes.html#inheritance).
- [The Python 2.3 Method Resolution Order](https://docs.python.org/3/howto/mro.html), the official explanation of the MRO algorithm (advanced).
- scikit-learn's baseline models: [`DummyClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html).

**For self-study.** Where to go next if you want to learn more:

- [Real Python: Inheritance and Composition, a Python OOP Guide](https://realpython.com/inheritance-composition-python/) (article, intermediate): when to use which, with a longer worked example.
- [Real Python: Supercharge Your Classes With Python super()](https://realpython.com/python-super/) (article, intermediate).
- [Raymond Hettinger, *Python's super() considered super!*](https://rhettinger.wordpress.com/2011/05/26/super-considered-super/) (blog post, advanced): how `super()` really works with multiple inheritance.

---

[← 01 · Classes and objects](01_classes_and_objects.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [03 · Special methods →](03_special_methods.ipynb)